# Prueba de publicación

In [7]:
import os
import json
import time
import pandas as pd
from google.cloud import pubsub_v1

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "credentials.json"

project_id = "bueno-398402"
topic_id = "ecommerce-topic"

publisher = pubsub_v1.PublisherClient()
topic_path = publisher.topic_path(project_id, topic_id)

print("Leyendo dataset local...")
df = pd.read_csv("data/data_ecommerce.csv", encoding="ISO-8859-1")

# Tomamos las últimas 20 filas para simular eventos nuevos en vivo sin gastar saldo
df_stream = df.tail(20)

print("Iniciando simulación de streaming en vivo...")

for index, row in df_stream.iterrows():
    payload = {
        "InvoiceNo": str(row["InvoiceNo"]),
        "StockCode": str(row["StockCode"]),
        "Description": str(row["Description"]),
        "Quantity": int(row["Quantity"]),
        "InvoiceDate": time.strftime("%Y-%m-%d %H:%M:%S"),
        "UnitPrice": float(row["UnitPrice"]),
        "CustomerID": str(int(row["CustomerID"])) if pd.notnull(row["CustomerID"]) else "00000",
        "Country": str(row["Country"])
    }

    data = json.dumps(payload).encode("utf-8")
    future = publisher.publish(topic_path, data)
    message_id = future.result()

    print(
        f"Evento enviado -> Message ID: {message_id} | "
        f"Factura: {payload['InvoiceNo']} | "
        f"Hora: {payload['InvoiceDate']}"
    )

    time.sleep(3)

print("¡Simulación terminada con éxito!")

Leyendo dataset local...
Iniciando simulación de streaming en vivo...
Evento enviado -> Message ID: 19078038016006337 | Factura: 581585 | Hora: 2026-05-17 12:20:44
Evento enviado -> Message ID: 19078159915202479 | Factura: 581586 | Hora: 2026-05-17 12:20:48
Evento enviado -> Message ID: 19076871754688754 | Factura: 581586 | Hora: 2026-05-17 12:20:51
Evento enviado -> Message ID: 19077285823287778 | Factura: 581586 | Hora: 2026-05-17 12:20:54
Evento enviado -> Message ID: 19077255451811878 | Factura: 581586 | Hora: 2026-05-17 12:20:57
Evento enviado -> Message ID: 19076606344465814 | Factura: 581587 | Hora: 2026-05-17 12:21:00
Evento enviado -> Message ID: 19077381871725880 | Factura: 581587 | Hora: 2026-05-17 12:21:03
Evento enviado -> Message ID: 19077236137805444 | Factura: 581587 | Hora: 2026-05-17 12:21:06
Evento enviado -> Message ID: 19078264941495104 | Factura: 581587 | Hora: 2026-05-17 12:21:09
Evento enviado -> Message ID: 19077489571554475 | Factura: 581587 | Hora: 2026-05-17

# Streaming Masivo

Para creación del dashboard.

In [5]:
import os
import json
import time
import pandas as pd
from google.cloud import pubsub_v1

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "credentials.json"

project_id = "bueno-398402"
topic_id = "ecommerce-topic"

publisher = pubsub_v1.PublisherClient()
topic_path = publisher.topic_path(project_id, topic_id)

print("Leyendo dataset local...")
df = pd.read_csv("data/data_ecommerce.csv", encoding="ISO-8859-1")

# Muestra de 10,000 filas
df_stream = df.tail(10000).copy()

print(f"Iniciando publicación de {len(df_stream)} eventos...")

futures = []
start_time = time.time()

for i, (_, row) in enumerate(df_stream.iterrows(), start=1):

    payload = {
        "InvoiceNo": str(row["InvoiceNo"]),
        "StockCode": str(row["StockCode"]),
        "Description": str(row["Description"]),
        "Quantity": int(row["Quantity"]),
        "InvoiceDate": time.strftime("%Y-%m-%d %H:%M:%S"),
        "UnitPrice": float(row["UnitPrice"]),
        "CustomerID": str(int(row["CustomerID"])) if pd.notnull(row["CustomerID"]) else "00000",
        "Country": str(row["Country"])
    }

    data = json.dumps(payload).encode("utf-8")
    future = publisher.publish(topic_path, data)
    futures.append(future)

    if i % 500 == 0:
        print(f"{i} eventos publicados...")
        time.sleep(0.2)

# Confirmar publicación
print("Confirmando publicación de mensajes...")
for future in futures:
    future.result()

elapsed = round(time.time() - start_time, 2)
print(f"Simulación terminada. Total publicado: {len(futures)} eventos en {elapsed} segundos.")

Leyendo dataset local...
Iniciando publicación de 10000 eventos...
500 eventos publicados...
1000 eventos publicados...
1500 eventos publicados...
2000 eventos publicados...
2500 eventos publicados...
3000 eventos publicados...
3500 eventos publicados...
4000 eventos publicados...
4500 eventos publicados...
5000 eventos publicados...
5500 eventos publicados...
6000 eventos publicados...
6500 eventos publicados...
7000 eventos publicados...
7500 eventos publicados...
8000 eventos publicados...
8500 eventos publicados...
9000 eventos publicados...
9500 eventos publicados...
10000 eventos publicados...
Confirmando publicación de mensajes...
Simulación terminada. Total publicado: 10000 eventos en 5.05 segundos.


# Ingesta final

Para mostrar que el dashboard toma registros recientes.

In [6]:
import os
import json
import time
import random
import pandas as pd
from datetime import datetime, timedelta
from google.cloud import pubsub_v1

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "credentials.json"

project_id = "bueno-398402"
topic_id = "ecommerce-topic"

publisher = pubsub_v1.PublisherClient()
topic_path = publisher.topic_path(project_id, topic_id)

print("Leyendo dataset local...")
df = pd.read_csv("data/data_ecommerce.csv", encoding="ISO-8859-1")

# Filtrar países
df_germany = df[df["Country"] == "Germany"].copy()
df_netherlands = df[df["Country"] == "Netherlands"].copy()

print(f"Filas disponibles Germany: {len(df_germany)}")
print(f"Filas disponibles Netherlands: {len(df_netherlands)}")

# Tomar muestra. Si no hay suficientes filas, usa reemplazo.
sample_germany = df_germany.sample(
    n=7000,
    replace=len(df_germany) < 7000,
    random_state=42
)

sample_netherlands = df_netherlands.sample(
    n=3000,
    replace=len(df_netherlands) < 3000,
    random_state=42
)

df_stream = pd.concat([sample_germany, sample_netherlands], ignore_index=True)

# Mezclar para que lleguen intercalados
df_stream = df_stream.sample(frac=1, random_state=42).reset_index(drop=True)

# Rango aleatorio para la serie temporal
start_date = datetime(2026, 5, 17, 8, 0, 0)
end_date = datetime(2026, 5, 17, 18, 0, 0)

def random_datetime(start, end):
    delta = end - start
    random_seconds = random.randint(0, int(delta.total_seconds()))
    return start + timedelta(seconds=random_seconds)

print(f"Iniciando publicación de {len(df_stream)} eventos balanceados...")

futures = []
start_time = time.time()

for i, (_, row) in enumerate(df_stream.iterrows(), start=1):

    random_invoice_date = random_datetime(start_date, end_date)

    payload = {
        "InvoiceNo": str(row["InvoiceNo"]),
        "StockCode": str(row["StockCode"]),
        "Description": str(row["Description"]),
        "Quantity": int(row["Quantity"]),
        "InvoiceDate": random_invoice_date.strftime("%Y-%m-%d %H:%M:%S"),
        "UnitPrice": float(row["UnitPrice"]),
        "CustomerID": str(int(row["CustomerID"])) if pd.notnull(row["CustomerID"]) else "00000",
        "Country": str(row["Country"])
    }

    data = json.dumps(payload).encode("utf-8")
    future = publisher.publish(topic_path, data)
    futures.append(future)

    if i % 500 == 0:
        print(f"{i} eventos publicados...")
        time.sleep(0.2)

print("Confirmando publicación de mensajes...")
for future in futures:
    future.result()

elapsed = round(time.time() - start_time, 2)
print(f"Simulación terminada. Total publicado: {len(futures)} eventos en {elapsed} segundos.")

Leyendo dataset local...
Filas disponibles Germany: 9495
Filas disponibles Netherlands: 2371
Iniciando publicación de 10000 eventos balanceados...
500 eventos publicados...
1000 eventos publicados...
1500 eventos publicados...
2000 eventos publicados...
2500 eventos publicados...
3000 eventos publicados...
3500 eventos publicados...
4000 eventos publicados...
4500 eventos publicados...
5000 eventos publicados...
5500 eventos publicados...
6000 eventos publicados...
6500 eventos publicados...
7000 eventos publicados...
7500 eventos publicados...
8000 eventos publicados...
8500 eventos publicados...
9000 eventos publicados...
9500 eventos publicados...
10000 eventos publicados...
Confirmando publicación de mensajes...
Simulación terminada. Total publicado: 10000 eventos en 5.09 segundos.


# Prueba de Modelo ML

In [11]:
import json
from google.cloud import aiplatform

project = "867022140217"
endpoint_id = "5252543517502210048"
location = "us-central1"

aiplatform.init(project=project, location=location)

endpoint = aiplatform.Endpoint(
    endpoint_name=f"projects/{project}/locations/{location}/endpoints/{endpoint_id}"
)

instances = [
    {
        "Quantity": 1,
        "UnitPrice": 9.99,
        "Country": "Germany"
    }
]

prediction = endpoint.predict(instances=instances)

# Convertir la respuesta a un diccionario más legible
output = {
    "input": instances[0],
    "prediction": prediction.predictions[0],
    "deployed_model_id": prediction.deployed_model_id,
    "model_resource_name": prediction.model_resource_name,
    "model_version_id": prediction.model_version_id
}

print(json.dumps(output, indent=4, ensure_ascii=False))

{
    "input": {
        "Quantity": 1,
        "UnitPrice": 9.99,
        "Country": "Germany"
    },
    "prediction": {
        "IsReturn_probs": [
            0.5515341763211642,
            0.4484658236788358
        ],
        "predicted_IsReturn": [
            "1"
        ],
        "IsReturn_values": [
            "1",
            "0"
        ]
    },
    "deployed_model_id": "5134095328865157120",
    "model_resource_name": "projects/867022140217/locations/us-central1/models/return-prediction-bqml",
    "model_version_id": "1"
}


In [12]:
pred = prediction.predictions[0]

print("Resultado de inferencia en Vertex AI")
print("-----------------------------------")
print(f"Entrada: {instances[0]}")
print(f"Predicción IsReturn: {pred['predicted_IsReturn'][0]}")
print(f"Probabilidades: {pred['IsReturn_probs']}")
print(f"Clases: {pred['IsReturn_values']}")

Resultado de inferencia en Vertex AI
-----------------------------------
Entrada: {'Quantity': 1, 'UnitPrice': 9.99, 'Country': 'Germany'}
Predicción IsReturn: 1
Probabilidades: [0.5515341763211642, 0.4484658236788358]
Clases: ['1', '0']


Se probó el endpoint de Vertex AI usando una instancia con las variables Quantity, UnitPrice y Country, que son las columnas de entrada del modelo. El endpoint respondió correctamente con la predicción predicted_IsReturn = 1 y las probabilidades asociadas, validando la inferencia online del modelo desplegado.